In [ ]:
from pymongo import MongoClient

HOST = "10.255.68.40"
PORT = 27017
DB_NAME = "ejoow"

POIS_COL_NAME = "1. US_pois"
CHECKIN_COL_NAME = "2. US_checkin_v2"
CHECKIN_DIST_COL_NAME = "2. US_checkin_tl_distribution"
POIS_DIST_COL_NAME = "2. US_pois_distribution"

client = MongoClient(f"mongodb://{HOST}:{PORT}/")
db = client[DB_NAME]

pois_col = db[POIS_COL_NAME]
checkin_col = db[CHECKIN_COL_NAME]
checkin_dist_col = db[CHECKIN_DIST_COL_NAME]
pois_dist_col = db[POIS_DIST_COL_NAME]

checkin_col.create_index("venue_id")
pois_col.create_index("venue_id")

# 1) Aggregate TL visit counts for each venue_id.
pipeline_checkin_dist = [
    {
        "$match": {
            "venue_id": {"$exists": True, "$ne": None},
            "TL": {"$in": [1, 2, 3, 4]},
        }
    },
    {
        "$group": {
            "_id": "$venue_id",
            "loc_cluster_id_k5": {"$first": "$loc_cluster_id_k5"},
            "visit1": {"$sum": {"$cond": [{"$eq": ["$TL", 1]}, 1, 0]}},
            "visit2": {"$sum": {"$cond": [{"$eq": ["$TL", 2]}, 1, 0]}},
            "visit3": {"$sum": {"$cond": [{"$eq": ["$TL", 3]}, 1, 0]}},
            "visit4": {"$sum": {"$cond": [{"$eq": ["$TL", 4]}, 1, 0]}}
        }
    },
    {
        "$project": {
            "_id": 0,
            "venue_id": "$_id",
            "loc_cluster_id_k5": 1,
            "visit1": 1,
            "visit2": 1,
            "visit3": 1,
            "visit4": 1
        }
    },
    {
        "$out": CHECKIN_DIST_COL_NAME
    }
]

db.command(
    "aggregate",
    CHECKIN_COL_NAME,
    pipeline=pipeline_checkin_dist,
    cursor={},
    allowDiskUse=True,
)

checkin_dist_col.create_index("venue_id", unique=True)

# 2) Attach loc_cluster_id_k5 and visit1~visit4 to POI metadata and write the final collection.
pipeline_pois_dist = [
    {
        "$lookup": {
            "from": CHECKIN_DIST_COL_NAME,
            "localField": "venue_id",
            "foreignField": "venue_id",
            "as": "dist"
        }
    },
    {
        "$unwind": {
            "path": "$dist",
            "preserveNullAndEmptyArrays": True
        }
    },
    {
        "$project": {
            "_id": 1,
            "venue_id": 1,
            "latitude": 1,
            "longitude": 1,
            "category_id": 1,
            "category": 1,
            "country": 1,
            "loc_cluster_id_k5": "$dist.loc_cluster_id_k5",
            "visit1": {"$ifNull": ["$dist.visit1", 0]},
            "visit2": {"$ifNull": ["$dist.visit2", 0]},
            "visit3": {"$ifNull": ["$dist.visit3", 0]},
            "visit4": {"$ifNull": ["$dist.visit4", 0]}
        }
    },
    {
        "$out": POIS_DIST_COL_NAME
    }
]

db.command(
    "aggregate",
    POIS_COL_NAME,
    pipeline=pipeline_pois_dist,
    cursor={},
    allowDiskUse=True,
)

pois_dist_col.create_index("venue_id")

print("Done.")
print("checkin tl distribution count:", checkin_dist_col.estimated_document_count())
print("poi distribution count:", pois_dist_col.estimated_document_count())
print("sample with visits:")
print(
    pois_dist_col.find_one(
        {
            "$or": [
                {"visit1": {"$gt": 0}},
                {"visit2": {"$gt": 0}},
                {"visit3": {"$gt": 0}},
                {"visit4": {"$gt": 0}},
            ]
        }
    )
)


In [ ]:
category_count_result = list(
    pois_dist_col.aggregate(
        [
            {
                "$group": {
                    "_id": {
                        "category_id": "$category_id",
                        "category": "$category",
                    }
                }
            },
            {
                "$count": "total_categories"
            },
        ]
    )
)

total_categories = category_count_result[0]["total_categories"] if category_count_result else 0
print("Total distinct categories:", total_categories)
